In [1]:
import os
import pandas as pd 
import numpy as np
from lets_plot import * # This imports all of ggplot2's functions
LetsPlot.setup_html()
import matplotlib.pyplot as plt

In [2]:
# List all files in the ME204/data/waitrose folder
all_files = [os.path.join('E:/DE/me204/data/scrap', file) for file in os.listdir('E:/DE/me204/data/scrap') 
             if file.endswith('.csv')]

# Read every single file and concatenate them into a single DataFrame with pandas concat
df = pd.concat((pd.read_csv(file) for file in all_files))
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 492 entries, 0 to 22
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Name              492 non-null    object 
 1   Discount          492 non-null    object 
 2   Price             492 non-null    object 
 3   Item No.          492 non-null    int64  
 4   Material          492 non-null    object 
 5   Designer          492 non-null    object 
 6   Warranty          492 non-null    object 
 7   Price_cleaned     196 non-null    float64
 8   Discount_cleaned  196 non-null    float64
dtypes: float64(2), int64(1), object(6)
memory usage: 38.4+ KB


In [3]:
df = df.drop_duplicates()

In [4]:
df

,Name,Discount,Price,Item No.,Material,Designer,Warranty,Price_cleaned,Discount_cleaned
0,Hot Dog Picnic Poster,25% off,$671.25,100487604,American maple frame,Steve Frykholm,1-year warranty,671.25,0.25
1,Girard Geometric E,25% off,$146.25,100400155,100% cotton paper,Alexander Girard,1-year warranty,146.25,0.25
2,Deluxx Visions Print,20% off,$200.00,100606845,Archival pigment ink,FAILE and Deluxx Fluxx,1-year warranty,200.00,0.20
3,Nelson Pop Art Blue and Black Poster,25% off,$408.75,100291491,"American maple frame with natural, white, or b...",George Nelson,1-year warranty,408.75,0.25
4,Come Fly With Me Print,20% off,$200.00,100606844,Archival pigment ink,FAILE and Deluxx Fluxx,1-year warranty,200.00,0.20
...,...,...,...,...,...,...,...,...,...
18,Eames Storage Unit,25% off,$971.25,100074720,Molded plywood shelves,Charles and Ray Eames,5-year warranty,NaN,NaN
19,Eames Storage Unit,25% off,$596.25,100074719,Molded plywood shelves,Charles and Ray Eames,5-year warranty,NaN,NaN
20,Nelson Thin Edge 3-Drawer Chest,25% off,"$3,746.25",100145137,"Walnut, white ash, or santos palisander veneer...",George Nelson,5-year warranty,NaN,NaN
21,Nelson Thin Edge Double Dresser,25% off,"$5,621.25",10002007,"Walnut, white ash or santos palisander veneer ...",George Nelson,5-year warranty,NaN,NaN


data cleaning using a function remove the str in the price and discount for the further calculation

In [5]:
def clean_price(price: str):
    """
    Cleans the price string by removing currency symbols and formatting it as a float.

    Parameters:
    price (str): The price value as a string.

    Returns:
    float: The cleaned price.
    """
    if isinstance(price, str):
        price = price.replace("$", "").replace("£", "").replace(",", "").strip()
    return float(price) if price.replace(".", "").isdigit() else None

def clean_discount(discount: str):
    """
    Extracts the numeric discount percentage from the discount column.

    Parameters:
    discount (str): The discount value as a string.

    Returns:
    float: The numeric discount percentage.
    """
    if isinstance(discount, str) and "%" in discount:
        return float(discount.replace("% off", "").strip()) / 100  # Converts to decimal (e.g., 25% → 0.25)
    return None

# Apply cleaning functions
df["Price_cleaned"] = df["Price"].astype(str).apply(clean_price)
df["Discount_cleaned"] = df["Discount"].astype(str).apply(clean_discount)

# Display cleaned dataframe
df

,Name,Discount,Price,Item No.,Material,Designer,Warranty,Price_cleaned,Discount_cleaned
0,Hot Dog Picnic Poster,25% off,$671.25,100487604,American maple frame,Steve Frykholm,1-year warranty,671.25,0.25
1,Girard Geometric E,25% off,$146.25,100400155,100% cotton paper,Alexander Girard,1-year warranty,146.25,0.25
2,Deluxx Visions Print,20% off,$200.00,100606845,Archival pigment ink,FAILE and Deluxx Fluxx,1-year warranty,200.00,0.20
3,Nelson Pop Art Blue and Black Poster,25% off,$408.75,100291491,"American maple frame with natural, white, or b...",George Nelson,1-year warranty,408.75,0.25
4,Come Fly With Me Print,20% off,$200.00,100606844,Archival pigment ink,FAILE and Deluxx Fluxx,1-year warranty,200.00,0.20
...,...,...,...,...,...,...,...,...,...
18,Eames Storage Unit,25% off,$971.25,100074720,Molded plywood shelves,Charles and Ray Eames,5-year warranty,971.25,0.25
19,Eames Storage Unit,25% off,$596.25,100074719,Molded plywood shelves,Charles and Ray Eames,5-year warranty,596.25,0.25
20,Nelson Thin Edge 3-Drawer Chest,25% off,"$3,746.25",100145137,"Walnut, white ash, or santos palisander veneer...",George Nelson,5-year warranty,3746.25,0.25
21,Nelson Thin Edge Double Dresser,25% off,"$5,621.25",10002007,"Walnut, white ash or santos palisander veneer ...",George Nelson,5-year warranty,5621.25,0.25


convert the cleaned data into csv

In [6]:
import pandas as pd

# Specify the correct file path (with .csv extension)
storage_path = r"E:/DE/me204/data/scrap/cleaned_data.csv"  # Ensure this is a valid directory

# Save DataFrame as CSV
df.to_csv(storage_path, index=False)

print(f"CSV file saved at: {storage_path}")


CSV file saved at: E:/DE/me204/data/scrap/cleaned_data.csv


# SQL LOADING 

In [7]:
import pandas as pd
import sqlite3
import os

# Define file paths
csv_path = r"E:/DE/me204/data/scrap/cleaned_data.csv"  
db_path = r"E:/DE/me204/data/clean/cleaned_data.db"  

# Ensure directory exists
os.makedirs(os.path.dirname(db_path), exist_ok=True)

# Load CSV into a DataFrame
df = pd.read_csv(csv_path)

# Connect to SQLite database
conn = sqlite3.connect(db_path)

# Create Main Table
df.to_sql("main_data", conn, if_exists="replace", index=False)

# Create Discount Table (Only Name & Discount)
df[['Item No.','Discount_cleaned']].to_sql("discount_table", conn, if_exists="replace", index=False)

# Create Designer Table (Only Name & Designer)
df[['Item No.', 'Designer']].to_sql("designer_table", conn, if_exists="replace", index=False)

# Close Connection
conn.close()

print(f"Tables created in SQLite at: {db_path}")


Tables created in SQLite at: E:/DE/me204/data/clean/cleaned_data.db
